## 2. Data understanding (Wil)

### Load the data

In [ ]:
from google.colab import drive                       # tool to connect Google Drive to Colab
drive.mount("/content/drive")                        # connects your Drive (Colab asks you to allow access)
import pandas as pd
imported_data = pd.read_csv("/content/drive/MyDrive/GESTS420/Hotel Reservations.csv")   # reads the CSV from your GESTS420 folder

### D1. Overview

In [2]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which requested element it covers]
du_raw = imported_data.copy()                               # copy of the data exactly as loaded in cell 0 -> [used by all elements below]
print("Rows (bookings):", du_raw.shape[0])                 # number of rows; one row = one booking -> [3. Overview]
print("Columns (variables):", du_raw.shape[1])             # number of columns -> [3. Overview]
du_date_cols = du_raw.columns[du_raw.columns.str.startswith("arrival_")]   # finds the columns whose name starts with "arrival_" (year, month, day) -> [3. Overview + 4. Variable types]
du_arrival = pd.to_datetime(                                # builds one real date from these three columns -> [3. Overview + 9. Impossible dates]
    du_raw[du_date_cols].set_axis(["year", "month", "day"], axis=1),   # renames them year/month/day so pandas knows which is which -> [3. Overview]
    errors="coerce")                                        # a date that does not exist becomes NaT (empty) instead of an error -> [9. Impossible dates]
print("Arrivals from", du_arrival.min().date(), "to", du_arrival.max().date())   # first and last arrival date = period covered -> [3. Overview]

Rows (bookings): 36275
Columns (variables): 19
Arrivals from 2017-07-01 to 2018-12-31


### D2. Variable types and modalities

In [3]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which requested element it covers]
du_modalities = du_raw.nunique()                            # number of different values in each column -> [1. Modalities]
du_identifier = du_raw.columns[du_modalities == len(du_raw)]   # columns with as many different values as rows = identifiers -> [4. Variable types]
du_text = du_raw.select_dtypes("object").columns.drop(du_identifier)   # text columns, without the identifier -> [4. Variable types]
du_binary = du_raw.columns[du_raw.isin([0, 1]).all()]       # columns where every value is 0 or 1 (no/yes) -> [4. Variable types]
du_categorical = du_text.append(du_binary)                  # categorical = text columns + 0/1 columns -> [4. Variable types]
du_numerical = du_raw.select_dtypes("number").columns.drop(du_binary).drop(du_date_cols)   # numerical = the other number columns -> [4. Variable types]
du_overview = pd.DataFrame({"stored as": du_raw.dtypes,      # how pandas stores each column (int64, float64 or object) -> [4. Variable types]
                            "modalities": du_modalities})  # number of different values -> [1. Modalities]
du_overview["type"] = "Numerical"                           # start by calling every column numerical -> [4. Variable types]
du_overview.loc[du_categorical, "type"] = "Categorical"     # then overwrite the categorical ones -> [4. Variable types]
du_overview.loc[du_date_cols, "type"] = "Date part"         # the date parts -> [4. Variable types]
du_overview.loc[du_identifier, "type"] = "Identifier"       # and the identifier -> [4. Variable types]
du_overview                                                 # one line per attribute, in the original column order -> [1. Modalities + 4. Variable types]

,stored as,modalities,type
Booking_ID,object,36275,Identifier
no_of_adults,int64,5,Numerical
no_of_children,int64,6,Numerical
no_of_weekend_nights,int64,8,Numerical
no_of_week_nights,int64,18,Numerical
type_of_meal_plan,object,4,Categorical
required_car_parking_space,int64,2,Categorical
room_type_reserved,object,7,Categorical
lead_time,int64,352,Numerical
arrival_year,int64,2,Date part


### D3. Attributes, modalities and mode of the categorical variables

In [4]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which requested element it covers]
du_rows = []                                                # empty list that will hold one line per variable -> [2. Mode]
for du_col in du_categorical:                               # goes through each categorical variable found in D2 -> [2. Mode]
    du_counts = du_raw[du_col].value_counts()               # number of bookings per value, most frequent first -> [2. Mode]
    du_rows.append({"variable": du_col,                     # name of the variable -> [2. Mode]
                    "attributes": list(du_counts.index),    # every category found in the column, most frequent first -> [1. Modalities]
                    "modalities": du_modalities[du_col],    # number of different categories -> [1. Modalities]
                    "mode": du_counts.index[0],             # the most frequent value = the mode -> [2. Mode]
                    "share of the mode": round(du_counts.iloc[0] / len(du_raw), 3)})   # share of bookings that have the mode -> [2. Mode]
pd.DataFrame(du_rows).set_index("variable")                 # shows the results as a table -> [1. Modalities + 2. Mode]

,attributes,modalities,mode,share of the mode
variable,,,,
type_of_meal_plan,"[Meal Plan 1, Not Selected, Meal Plan 2, Meal ...",4,Meal Plan 1,0.767
room_type_reserved,"[Room_Type 1, Room_Type 4, Room_Type 6, Room_T...",7,Room_Type 1,0.775
market_segment_type,"[Online, Offline, Corporate, Complementary, Av...",5,Online,0.640
booking_status,"[Not_Canceled, Canceled]",2,Not_Canceled,0.672
required_car_parking_space,"[0, 1]",2,0,0.969
repeated_guest,"[0, 1]",2,0,0.974


### D4. Target variable

In [5]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which requested element it covers]
du_raw["booking_status"].value_counts(normalize=True).round(3)   # share of cancelled and not cancelled bookings (booking_status is the target of our research question) -> [5. Target]

booking_status
Not_Canceled    0.672
Canceled        0.328
Name: proportion, dtype: float64

### D5. Duplicates and impossible dates

In [6]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which requested element it covers]
du_twins = du_raw.drop(columns=du_identifier).duplicated(keep=False)   # True for EVERY row identical to another one apart from the identifier -> [8. Duplicates]
print("Rows identical to another row apart from the identifier:", du_twins.sum())   # counts them (the existing cell counts only the extra copies) -> [8. Duplicates]
du_invalid = du_arrival.isna()                              # True where the arrival date does not exist in the calendar -> [9. Impossible dates]
print("Impossible arrival dates:", du_invalid.sum())       # counts them -> [9. Impossible dates]
du_raw.loc[du_invalid, du_date_cols].value_counts()         # shows which dates they are -> [9. Impossible dates]

Rows identical to another row apart from the identifier: 13413
Impossible arrival dates: 37


arrival_year  arrival_month  arrival_date
2018          2              29              37
Name: count, dtype: int64

### D6. Cancellation rate by lead time

In [ ]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which slide or element it covers]
import matplotlib.pyplot as plt                                                # library to draw charts -> [lead time chart]
du_cancelled = (du_raw["booking_status"] == "Canceled").astype(int)            # 1 = cancelled, 0 = not cancelled -> [lead time chart]
du_bins = [-1, 7, 30, 60, 90, 150, 250, du_raw["lead_time"].max()]             # limits of the lead time groups, in days -> [lead time chart]
du_labels = ["0-7 d", "8-30 d", "31-60 d", "61-90 d", "91-150 d", "151-250 d", "251+ d"]   # names of the 7 groups -> [lead time chart]
du_lead_group = pd.cut(du_raw["lead_time"], bins=du_bins, labels=du_labels)    # puts each booking in its lead time group -> [lead time chart]
du_rate = du_cancelled.groupby(du_lead_group, observed=True).mean() * 100      # % of cancelled bookings in each group -> [lead time chart]
print(du_rate.round(0))                                                         # the percentages shown on the slide -> [lead time chart]
ax = du_rate.plot.bar(rot=0, color="#1E75BE", figsize=(9, 4))                  # bar chart, one bar per group -> [lead time chart]
ax.set_title("Cancellation rate by booking lead time")                         # chart title -> [lead time chart]
ax.set_xlabel("Days between booking and arrival")                              # x axis name -> [lead time chart]
ax.set_ylabel("% of bookings cancelled")                                       # y axis name -> [lead time chart]
plt.show()                                                                      # shows the chart -> [lead time chart]

### D7. Chart: lead time distribution

In [ ]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which slide or element it covers]
du_lead = du_raw["lead_time"]                                                 # lead time of every booking, in days -> [chart 1: numerical distribution]
ax = du_lead.plot.hist(bins=40, color="#1E75BE", figsize=(6.6, 2.6))          # histogram: number of bookings per lead time range -> [chart 1]
ax.axvline(du_lead.mean(), color="black", linestyle="--",
           label=f"mean {du_lead.mean():.0f} days")                           # dashed line at the mean -> [chart 1 + 2. Mean]
ax.axvline(du_lead.median(), color="orange", linewidth=2,
           label=f"median {du_lead.median():.0f} days")                       # line at the median -> [chart 1 + 2. Median]
ax.set_title("Lead time: mean 85 vs median 57 days")                       # chart title -> [chart 1]
ax.set_xlabel("Days between booking and arrival")                             # x axis name -> [chart 1]
ax.set_ylabel("Bookings")                                                     # y axis name -> [chart 1]
ax.legend(handles=ax.get_lines())                                            # legend for the two lines only -> [chart 1]
plt.tight_layout()                                                            # avoids cut-off labels -> [chart 1]
plt.show()                                                                    # shows the chart -> [chart 1]

### D8. Chart: bookings per market segment

In [ ]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which slide or element it covers]
du_segment = du_raw["market_segment_type"].value_counts(normalize=True) * 100   # % of bookings per channel -> [chart 2: categorical distribution + 2. Mode]
du_segment = du_segment.sort_values()                                         # smallest first, so the biggest bar is on top -> [chart 2]
ax = du_segment.plot.barh(color="#1E75BE", figsize=(6.6, 2.6))                # horizontal bar chart, one bar per channel -> [chart 2]
for du_i, du_v in enumerate(du_segment):                                      # goes through each bar -> [chart 2]
    ax.text(du_v + 0.5, du_i, f"{du_v:.0f}%" if du_v >= 1 else f"{du_v:.1f}%", va="center")   # writes the % next to the bar -> [chart 2]
ax.set_title("Market segment: most bookings come online (the mode)")          # chart title -> [chart 2]
ax.set_xlabel("% of bookings")                                                # x axis name -> [chart 2]
ax.set_ylabel("")                                                             # no y axis name (the channels are written) -> [chart 2]
ax.set_xlim(0, 75)                                                            # room for the % labels -> [chart 2]
plt.tight_layout()                                                            # avoids cut-off labels -> [chart 2]
plt.show()                                                                    # shows the chart -> [chart 2]

### D9. Chart: correlation with cancellation

In [ ]:
#Wil - Data understanding. This is AI generated code! (Claude)
#Each comment says what the line does -> [which slide or element it covers]
du_num = du_raw[du_numerical].copy()                                          # the 9 numerical variables found in D2 -> [chart 3: 10. Correlation]
du_num["cancelled"] = (du_raw["booking_status"] == "Canceled").astype(int)   # target as 1/0 so it can be correlated -> [chart 3]
du_corr = du_num.corr()["cancelled"].drop("cancelled").sort_values()         # correlation of each variable with cancellation (same numbers as the heatmap) -> [chart 3]
du_colors = ["#1E75BE" if du_v > 0 else "#3F4A55" for du_v in du_corr]       # blue = more cancellations, dark = fewer -> [chart 3]
ax = du_corr.plot.barh(color=du_colors, figsize=(6.4, 4.9))                  # one bar per variable -> [chart 3]
ax.axvline(0, color="black", linewidth=0.8)                                  # zero line -> [chart 3]
for du_i, du_v in enumerate(du_corr):                                         # goes through each bar -> [chart 3]
    ax.text(du_v + (0.01 if du_v >= 0 else -0.01), du_i, f"{du_v:.2f}",
            va="center", ha="left" if du_v >= 0 else "right", fontsize=8)     # writes the value next to the bar -> [chart 3]
ax.set_title("Correlation with cancellation")                                 # chart title -> [chart 3]
ax.set_xlabel("correlation (-1 to 1)")                                        # x axis name -> [chart 3]
ax.set_xlim(-0.5, 0.6)                                                      # room for the value labels -> [chart 3]
plt.tight_layout()                                                            # avoids cut-off labels -> [chart 3]
plt.show()                                                                    # shows the chart -> [chart 3]